In [1]:
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torchcodec.decoders import AudioDecoder

from zerosyl.quantizer import FeatureQuantizer
from zerosyl.wavlm import WavLMEncoder
from zerosyl.zerosyl import ZeroSylPooler

In [2]:
wavlm = WavLMEncoder.from_unilm_checkpoint(
    "checkpoints/WavLM-Large.pt",
    num_layers=22,
).cuda()
pooler = ZeroSylPooler().cuda()
quantizer = FeatureQuantizer.from_pretrained("checkpoints/zerosyl-v040").cuda()

wavlm.eval()
pooler.eval()
quantizer.eval()

print(f"WavLM layers={wavlm.cfg.num_layers}")
print(
    f"output_layer={pooler.cfg.output_layer}, "
    f"boundary_layer={pooler.cfg.boundary_layer}, "
    f"win_size={pooler.cfg.win_size}, "
    f"prominence={pooler.cfg.prominence}"
)
print(
    f"quantizer clusters={quantizer.cfg.n_clusters}, "
    f"collapsed_vocab_size={quantizer.vocab_size}, "
    f"sil_token_id={quantizer.sil_token_id}"
)

WavLM layers=22
output_layer=22, boundary_layer=13, win_size=3, prominence=0.45
quantizer clusters=10000, collapsed_vocab_size=9116, sil_token_id=9115


In [3]:
def load_audio(path: str):
    return AudioDecoder(path, num_channels=1, sample_rate=16000).get_all_samples().data


wav1 = load_audio("data/sample.flac")
wav2 = load_audio("data/sample_long.flac")

print(wav1.shape)
print(wav2.shape)

torch.Size([1, 93680])
torch.Size([1, 562480])


In [4]:
@torch.inference_mode()
def encode(
    waveforms: torch.Tensor,
    lengths: torch.Tensor | None = None,
    normalize: bool = True,
    center_pad: bool = True,
    right_pad_to_hop: bool = True,
):
    enc = wavlm(
        waveforms,
        lengths=lengths,
        normalize=normalize,
        center_pad=center_pad,
        right_pad_to_hop=right_pad_to_hop,
    )
    pooled = pooler(enc.all_hidden_states, enc.seqlens)
    units = quantizer(pooled.segment_features)
    units = units.masked_fill(pooled.segment_pad_mask, -1)
    return units, pooled.segment_pad_mask

In [5]:
encode(wav1.cuda().unsqueeze(0))

(tensor([[9115, 9115, 9115,  530, 1566, 6573, 2896, 4462, 7828,  274,  473,  391,
          7851, 9009, 2938, 4195, 7546, 1335, 5165, 5462,  538, 1487, 6136, 2514,
          9066, 8783, 9115, 9115]], device='cuda:0'),
 tensor([[False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False, False, False]],
        device='cuda:0'))

In [6]:
HOP = wavlm.cfg.conv_hop_length
CENTER_PAD = (wavlm.cfg.conv_win_length - HOP) // 2


def preprocess(wav: torch.Tensor) -> torch.Tensor:
    wav = wav.reshape(-1)
    wav = F.layer_norm(wav, wav.shape)
    wav = F.pad(wav, (0, HOP - wav.size(0) % HOP))
    return F.pad(wav, (CENTER_PAD, CENTER_PAD))


def collate(waveforms: list[torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
    processed = [preprocess(w) for w in waveforms]
    lengths = torch.tensor([w.size(0) for w in processed], dtype=torch.long)
    return pad_sequence(processed, batch_first=True).unsqueeze(1), lengths


waveforms, lengths = collate([wav1, wav2])
print(waveforms.shape, lengths)

torch.Size([2, 1, 562640]) tensor([ 93840, 562640])


In [7]:
encode(
    waveforms.cuda(),
    lengths.cuda(),
    # collate already does this preprocessing
    normalize=False,
    center_pad=False,
    right_pad_to_hop=False,
)

(tensor([[9115, 9115, 9115,  530, 1566, 6573, 2896, 4462, 7828,  274,  473,  391,
          7851, 9009, 2938, 4195, 7546, 1335, 5165, 5462,  538, 1487, 6136, 2514,
          9066, 8783, 9115, 9115,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,
            -1],
         [9115, 4178, 8600, 6498, 9115, 3259, 4118,  833, 8899, 3918, 3148, 1290,